In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

from plot_utils import plot_advanced_comparison

In [ ]:
savedir = 'RPL_figures/4_tlt12'
if not os.path.exists(savedir):
    os.makedirs(savedir)

In [ ]:
method_names = {
    "RPLVIC": r"RPL$^{\text{VIC}}$",
    "IL": "IL",
    "sup": "Sup.",
    "rand": "Rand.",
    "RPL": "RPL"
}

method_colors = {
    'sup': 'black',
    'rand': 'gray',
    'RPLVIC': '#8e4774',
    'IL': '#888888',
    'RPL': '#994455'
}

In [ ]:
synset_names = [
    "airplane",
    "cap",
    "car",
    "chair",
    "earphone",
    "guitar",
    "laptop",
    "motorbike",
    "mug",
    "piano",
    "telephone",
    "watercraft"
]

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    'text.usetex': True,          # Use LaTeX for r'' strings
    'font.family': 'sans-serif',  # Default font family for regular text
    'mathtext.fontset': 'dejavusans',  # Font for math not using r''
    'axes.formatter.use_mathtext': False  # Enable math text for tick labels
})
mpl.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8

In [ ]:
from cmcrameri import cm
from matplotlib.colors import ListedColormap


# generate color map for synsets from colorbrewer qualitative set
synset_colors = ["#8dd3c7", "#ffffb3", "#bebada", "#fb8072", "#80b1d3", "#fdb462", "#b3de69", "#fccde5", "#d9d9d9", "#bc80bd", "#ccebc5", "#ffed6f"]
cmap = ListedColormap(synset_colors)

In [ ]:
repr_dir = "tlt12_reprs"
threshold = 0.5

# Load results

In [ ]:
df_tlt12 = pd.read_csv(f"{repr_dir}/linear_results.csv")

In [ ]:
df_tlt12_thresholded_rot = pd.read_csv(f"{repr_dir}/linear_thresh_{threshold}_results.csv")

In [ ]:
per_synset_regr = pd.read_csv(f"{repr_dir}/linear_per_class_results.csv")

# Linear decoding performance

In [ ]:
df_tlt12.head()

In [ ]:
# add _th to the column names of the thresholded dataframe
thresh_col_names = [col + "_th" for col in df_tlt12_thresholded_rot.columns if col not in ["method", "split", "seed"]]
df_tlt12_thresholded_rot.columns = ["method", "split", "seed"] + thresh_col_names
df_tlt12_thresholded_rot.head()

In [ ]:
# Merge the thresholded dataframe with the original dataframe
df_tlt12 = df_tlt12.merge(df_tlt12_thresholded_rot, on=["method", "split", "seed"], how="left")

In [ ]:
test_df_tlt12 = df_tlt12[df_tlt12["split"] == "test"]
ood_test_df_tlt12 = df_tlt12[df_tlt12["split"] == "ood_test"]

In [ ]:
task_configs = {
    "synset": ["synset"],
    "Obj. pos.": ["trans_x", "trans_y", "trans_z"],
    "Obj. vel.": ["delta_trans_x", "delta_trans_y", "delta_trans_z"],
    "Obj. rot.": ["rot_z_sin", "rot_z_cos"],
    "Obj. rot.*": ["rot_z_sin_th", "rot_z_cos_th"],
    "Obj. rot. vel.": ["delta_rot_z_sin", "delta_rot_z_cos"],
    "Obj. rot. vel.*": ["delta_rot_z_sin_th", "delta_rot_z_cos_th"],
    "Spot theta": ["spot_theta"],
    "Spot phi": ["spot_phi_sin", "spot_phi_cos"],
    "Light hue": ["spot_hue_sin", "spot_hue_cos"],
    "Floor hue": ["floor_hue_sin", "floor_hue_cos"],
    }

In [ ]:
# convert test_df_tlt12 to a new dataframe with columns: dataset, method, task, seed, accuracy, r2
results_df = pd.DataFrame(columns=["dataset", "method", "task", "seed", "accuracy", "r2"])
for row in test_df_tlt12.itertuples():
    for task_name, task_cols in task_configs.items():
        if task_name == "synset":
            accuracy = row.synset
            r2 = np.nan
        else:
            accuracy = np.nan
            r2 = np.mean([getattr(row, col) for col in task_cols])
        results_df = pd.concat([results_df, pd.DataFrame({
            "dataset": ["3DLT"],
            "method": [row.method],
            "task": [task_name],
            "seed": [row.seed],
            "accuracy": [accuracy],
            "r2": [r2]
        })], ignore_index=True)
results_df.head()

In [ ]:
# orientation decoding performance for RPLVIC (all synsets)
results_df[(results_df["method"] == "RPLVIC") & (results_df["task"] == "Obj. rot.")]

In [ ]:
# orientation decoding performance for RPLVIC (earphones excluded)
results_df[(results_df["method"] == "RPLVIC") & (results_df["task"] == "Obj. rot.*")]

In [ ]:
metric_configs = {
    "accuracy": {
        "individual_tasks": {
            "synset": "Category",
        },
        "averaged_tasks": []
    },
    "r2": {
        "individual_tasks": {
            "Obj. pos.": r'$\boldsymbol{r}$',
            "Obj. vel.": r'$\boldsymbol{v}$',
            "Obj. rot.*": r'$\theta_3^*$',
            "Spot theta": r'$\theta^{\text{spot}}$',
            "Spot phi": r'$\boldsymbol{\phi}^{\text{spot}}$',
            "Light hue": "Spot hue",
            "Floor hue": "Floor hue"
        },
        "averaged_tasks": []
    }
}

In [ ]:
fig, ax = plot_advanced_comparison(
    results_df, methods_to_plot=["RPL", "RPLVIC", "IL"], baseline_methods=["sup", "rand"],
    metric_configs=metric_configs, method_names=method_names, method_colors=method_colors,
    figsize=(7.5, 1.), dataset='3DLT', point_size=10, sharey=True, plot_spacing=0.2, loc_legend=(1, 0.2)
)
fig.savefig(os.path.join(savedir, "3dlt_results_v24_thresholded.pdf"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# convert ood_test_df_tlt12 to a new dataframe with columns: dataset, method, task, seed, accuracy, r2
results_ood_df = pd.DataFrame(columns=["dataset", "method", "task", "seed", "accuracy", "r2"])
for row in ood_test_df_tlt12.itertuples():
    for task_name, task_cols in task_configs.items():
        if task_name == "synset":
            accuracy = row.synset
            r2 = np.nan
        else:
            accuracy = np.nan
            r2 = np.mean([getattr(row, col) for col in task_cols])
        results_ood_df = pd.concat([results_ood_df, pd.DataFrame({
            "dataset": ["3DLT"],
            "method": [row.method],
            "task": [task_name],
            "seed": [row.seed],
            "accuracy": [accuracy],
            "r2": [r2]
        })], ignore_index=True)
results_ood_df.head()

In [ ]:
# plot OOD results
fig, ax = plot_advanced_comparison(
    results_ood_df, methods_to_plot=["RPL", "RPLVIC", "IL"], baseline_methods=["sup", "rand"],
    metric_configs=metric_configs, method_names=method_names, method_colors=method_colors,
    figsize=(7.5, 1.), dataset='3DLT', point_size=10, sharey=True, plot_spacing=0.2, loc_legend=(1, 0.2)
)
fig.savefig(os.path.join(savedir, "3dlt_results_v24_ood_thresholded.pdf"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
task_configs_full = {
    "synset": ["synset"],
    r"$r_1$": ["trans_x"],
    r"$r_2$": ["trans_y"],
    r"$r_3$": ["trans_z"],
    r"$v_1$": ["delta_trans_x"],
    r"$v_2$": ["delta_trans_y"],
    r"$v_3$": ["delta_trans_z"],
    r"$\theta_1^*$": ["rot_x_th"],
    r"$\theta_2^*$": ["rot_y_th"],
    r"$\theta_3^*$": ["rot_z_sin_th", "rot_z_cos_th"],
    r"$\omega_1^*$": ["delta_rot_x_th"],
    r"$\omega_2^*$": ["delta_rot_y_th"],
    r"$\omega_3^*$": ["delta_rot_z_sin_th", "delta_rot_z_cos_th"],
    r"$\theta^{\text{spot}}$": ["spot_theta"],
    r"$\phi^{\text{spot}}$": ["spot_phi_sin", "spot_phi_cos"],
    "Light hue": ["spot_hue_sin", "spot_hue_cos"],
    "Floor hue": ["floor_hue_sin", "floor_hue_cos"],
    r"$\Delta\theta^{\text{spot}}$": ["delta_spot_theta"],
    r"$\Delta\phi^{\text{spot}}$": ["delta_spot_phi_sin", "delta_spot_phi_cos"],
    r"$\Delta\text{Light hue}$": ["delta_spot_hue_sin", "delta_spot_hue_cos"],
    r"$\Delta\text{Floor hue}$": ["delta_floor_hue_sin", "delta_floor_hue_cos"]
}

In [ ]:
# convert test_df_tlt12 to a new dataframe with columns: dataset, method, task, seed, accuracy, r2
full_results_df = pd.DataFrame(columns=["dataset", "method", "task", "seed", "accuracy", "r2"])
for row in test_df_tlt12.itertuples():
    for task_name, task_cols in task_configs_full.items():
        if task_name == "synset":
            accuracy = row.synset
            r2 = np.nan
        else:
            accuracy = np.nan
            r2 = np.mean([getattr(row, col) for col in task_cols])
        full_results_df = pd.concat([full_results_df, pd.DataFrame({
            "dataset": ["3DLT"],
            "method": [row.method],
            "task": [task_name],
            "seed": [row.seed],
            "accuracy": [accuracy],
            "r2": [r2]
        })], ignore_index=True)
full_results_df.head()

In [ ]:
# plot decoding performance for each individual translation variable
metric_configs = {
    "r2": {
        "individual_tasks": {
            r"$r_1$": r"$r_1$",
            r"$r_2$": r"$r_2$",
            r"$r_3$": r"$r_3$",
            r"$v_1$": r"$v_1$",
            r"$v_2$": r"$v_2$",
            r"$v_3$": r"$v_3$",
        },
        "averaged_tasks": []
    }
}
fig, ax = plot_advanced_comparison(
    full_results_df, methods_to_plot=["RPL", "RPLVIC", "IL"], baseline_methods=["rand"],
    metric_configs=metric_configs, method_names=method_names, method_colors=method_colors,
    figsize=(3.4, 0.8), dataset='3DLT', point_size=5, sharey=True, plot_spacing=0.2, loc_legend=(1, 0.2), display_legend=False
)
# for a in ax:
#     a.yaxis.labelpad = 0.2
fig.suptitle("Translation", fontsize=10, y=1.05)
fig.savefig(os.path.join(savedir, "3dlt_extended_results_v24_translation_wRPL.pdf"),
            bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# plot decoding performance for light and floor variables
metric_configs = {
    "r2": {
        "individual_tasks": {
            r"$\theta^{\text{spot}}$": r"$\theta^{\text{spot}}$",
            r"$\phi^{\text{spot}}$": r"$\phi^{\text{spot}}$",
            r"$\Delta\theta^{\text{spot}}$": r"$\Delta\theta^{\text{spot}}$",
            r"$\Delta\phi^{\text{spot}}$": r"$\Delta\phi^{\text{spot}}$",
            r"$\Delta\text{Light hue}$": r"$\Delta\text{Hue}^{\text{spot}}$",
            r"$\Delta\text{Floor hue}$": r"$\Delta\text{Hue}^{\text{floor}}$"
        },
        "averaged_tasks": []
    }
}
fig, ax = plot_advanced_comparison(
    full_results_df, methods_to_plot=["RPL", "RPLVIC", "IL"], baseline_methods=["rand"],
    metric_configs=metric_configs, method_names=method_names, method_colors=method_colors,
    figsize=(3.45, 0.8), dataset='3DLT', point_size=5, sharey=True, plot_spacing=0.2, loc_legend=(1, 0.2), ylim=(0, 0.8)
)
# remove y-axis label
# ax[0].set_ylabel("")
fig.suptitle(r"Light and floor", fontsize=10, y=1.05)
# decrease space between ylabel and y-ticks
# for a in ax:
#     a.yaxis.labelpad = 0.2
fig.savefig(os.path.join(savedir, "3dlt_extended_results_v24_light_floor_small_wRPL.pdf"),
            bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# plot decoding performance for each individual rotation variable (earphones excluded)
metric_configs = {
    "r2": {
        "individual_tasks": {
            r"$\theta_1^*$": r"$\theta_1^*$",
            r"$\theta_2^*$": r"$\theta_2^*$",
            r"$\theta_3^*$": r"$\theta_3^*$",
            r"$\omega_1^*$": r"$\omega_1^*$",
            r"$\omega_2^*$": r"$\omega_2^*$",
            r"$\omega_3^*$": r"$\omega_3^*$",
        },
        "averaged_tasks": []
    }
}
fig, ax = plot_advanced_comparison(
    full_results_df, methods_to_plot=["RPLVIC", "IL"], baseline_methods=["rand"],
    metric_configs=metric_configs, method_names=method_names, method_colors=method_colors,
    figsize=(3.2, 0.8), dataset='3DLT', point_size=5, sharey=True, plot_spacing=0.2, loc_legend=(0.85, 0.45)
)
for a in ax:
    a.yaxis.labelpad = 0.2
ax[0].set_yticks([0.0, 0.1])
# fig.suptitle("Rotation", fontsize=10, y=1.05)
fig.savefig(os.path.join(savedir, "3dlt_extended_results_v24_rotation.pdf"),
            bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# plot theta_3 R2 values for all synsets vs earphones excluded
fig, ax = plt.subplots(1, 1, figsize=(1.2, 0.9), constrained_layout=True)
for method in results_df["method"].unique():
    if method not in ["IL", "RPLVIC"]:
        continue
    for seed in results_df["seed"].unique():
        r2_all = results_df[(results_df["method"] == method) & 
                            (results_df["seed"] == seed) & 
                            (results_df["task"] == "Obj. rot.")]["r2"].values[0]
        r2_thresholded = results_df[(results_df["method"] == method) & 
                                    (results_df["seed"] == seed) & 
                                    (results_df["task"] == "Obj. rot.*")]["r2"].values[0]
        ax.plot([0, 1], [r2_all, r2_thresholded], color=method_colors[method], 
                alpha=0.5, linewidth=1, marker='o', markersize=3, linestyle='--', label=method_names[method] if seed == 0 else "")
sns.despine()
ax.set_xticks([0, 1])
ax.set_xticklabels([r"$\theta_3$", r"$\theta_3^*$"], fontsize=8, rotation=0)
ax.set_ylabel(r"$R^2$", fontsize=8)
ax.set_yticks([0, 0.1])
ax.set_yticklabels([0, 0.1], fontsize=8)
ax.yaxis.labelpad = 0.2
ax.legend(loc='center left', bbox_to_anchor=(0.7, 0.45), frameon=False, fontsize=6, handletextpad=0.3)
fig.savefig(os.path.join(savedir, "3dlt_rot_z_all_vs_filtered.pdf"),
            bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# print summary of results for each method and task
summary = full_results_df.groupby(["method", "task"]).agg({"accuracy": ["mean", "std"], "r2": ["mean", "std"]}).reset_index()

summary = summary.round(3)
for method in summary["method"].unique():
    print(f"Method: {method}")
    for task in summary["task"].unique():
        row = summary[(summary["method"] == method) & (summary["task"] == task)]
        if row["accuracy"]["mean"].notnull().any():
            mean_acc = row["accuracy"]["mean"].values[0]
            std_acc = row["accuracy"]["std"].values[0]
            print(f"Task: {task}, Accuracy: {mean_acc:.2f} ± {std_acc:.2f}")
        if row["r2"]["mean"].notnull().any():
            mean_r2 = row["r2"]["mean"].values[0]
            std_r2 = row["r2"]["std"].values[0]
            print(f"Task: {task}, R2: {mean_r2:.2f} ± {std_r2:.2f}")
        

In [ ]:
# convert df_ood to a new dataframe with columns: dataset, method, task, seed, accuracy, r2
full_ood_results_df = pd.DataFrame(columns=["dataset", "method", "task", "seed", "accuracy", "r2"])
for row in ood_test_df_tlt12.itertuples():
    for task_name, task_cols in task_configs_full.items():
        if task_name == "synset":
            accuracy = row.synset
            r2 = np.nan
        else:
            accuracy = np.nan
            r2 = np.mean([getattr(row, col) for col in task_cols])
        full_ood_results_df = pd.concat([full_ood_results_df, pd.DataFrame({
            "dataset": ["3DLT"],
            "method": [row.method],
            "task": [task_name],
            "seed": [row.seed],
            "accuracy": [accuracy],
            "r2": [r2]
        })], ignore_index=True)
full_ood_results_df.head()

In [ ]:
# print summary of results for each method and task for OOD
summary_ood = full_ood_results_df.groupby(["method", "task"]).agg({"accuracy": ["mean", "std"], "r2": ["mean", "std"]}).reset_index()

summary_ood = summary_ood.round(3)
for method in summary_ood["method"].unique():
    print(f"Method: {method}")
    for task in summary_ood["task"].unique():
        row = summary_ood[(summary_ood["method"] == method) & (summary_ood["task"] == task)]
        if row["accuracy"]["mean"].notnull().any():
            mean_acc = row["accuracy"]["mean"].values[0]
            std_acc = row["accuracy"]["std"].values[0]
            print(f"Task: {task}, Accuracy: {mean_acc:.2f}\\pm{std_acc:.2f}")
        if row["r2"]["mean"].notnull().any():
            mean_r2 = row["r2"]["mean"].values[0]
            std_r2 = row["r2"]["std"].values[0]
            print(f"Task: {task}, R2: {mean_r2:.2f}\\pm{std_r2:.2f}")
        

In [ ]:
per_synset_regr_test = per_synset_regr[per_synset_regr["split"] == "test"]

In [ ]:
# convert per_synset_regr_test to a new dataframe with columns: dataset, method, task, seed, accuracy, r2
df_synsets = pd.DataFrame(columns=["dataset", "method", "task", "seed", "synset", "r2"])
for row in per_synset_regr_test.itertuples():
    for task_name, task_cols in task_configs.items():
        r2 = np.mean([getattr(row, col) for col in task_cols])
        df_synsets = pd.concat([df_synsets, pd.DataFrame({
            "dataset": ["3DLT"],
            "method": [row.method],
            "task": [task_name],
            "seed": [row.seed],
            "synset": [synset_names[row.synset]],
            "r2": [r2]
        })], ignore_index=True)
df_synsets.head()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(3.5, 0.8))
for i, synset_name in enumerate(synset_names):
    synset_data = df_synsets[df_synsets["synset"] == synset_name]
    synset_data = synset_data[synset_data["task"] == "Obj. rot."]
    for j, method in enumerate(method_names.keys()):
        if method in ["sup", "rand", "RPL"]:
            continue
        method_data = synset_data[synset_data["method"] == method]
        avg_value = method_data["r2"].mean()
        axes.scatter(i + np.random.uniform(-0.2, 0.2, len(method_data)), method_data["r2"], 
                     color=method_colors[method], s=10, alpha=0.5)
        axes.hlines(avg_value, xmin=i - 0.3, xmax=i + 0.3, color=method_colors[method], label=method_names[method] if i == 0 else "", linewidth=1)
axes.set_ylabel(r"$R^2\;(\theta_3)$", fontsize=8)
axes.set_xticks(range(len(synset_names)))
axes.set_xticklabels([s.capitalize() for s in synset_names], rotation=45, ha='right', fontsize=8)

handles, labels = axes.get_legend_handles_labels()
new_handles = []
for handle, label in zip(handles, labels):
    if label != "Threshold":
        new_handles.append(plt.Line2D([0], [0], marker='o', color='w', label=handle.get_label(),
                                    markerfacecolor=handle.get_color(), markersize=5))
    else:
        new_handles.append(handle)
axes.legend(new_handles, labels, loc='center left', bbox_to_anchor=(0.95, 0.75), frameon=False, fontsize=6, handletextpad=0.1)
sns.despine()
fig.savefig(os.path.join(savedir, "3dlt_rotz_v24_per_synset.pdf"), bbox_inches='tight', dpi=300, transparent=True)

# Load representations and labels

In [ ]:
# Change to the name you used to save the representations
rpl_filename = os.path.join(repr_dir, "RPLVIC_1.npz")
il_filename = os.path.join(repr_dir, "IL_1.npz")

In [ ]:
rpl_vic_1_data = np.load(rpl_filename)
X_train = rpl_vic_1_data["X_train"]
X_test = rpl_vic_1_data["X_test"]
y_test_clf = rpl_vic_1_data["y_test_clf"]
y_test_regr = rpl_vic_1_data["y_test_regr"]

In [ ]:
IL_1_data = np.load(il_filename)
X_train_IL = IL_1_data["X_train"]
X_test_IL = IL_1_data["X_test"]
y_test_clf_IL = IL_1_data["y_test_clf"]
y_test_regr_IL = IL_1_data["y_test_regr"]

In [ ]:
# plot colorbar for synsets
a = np.arange(len(synset_names)).reshape(1, -1)
plt.figure(figsize=(0.1, 1.8))
img = plt.imshow(a, cmap=cmap)
plt.gca().set_visible(False)
cax = plt.axes([0.1, 0.2, 0.8, 0.6])
plt.colorbar(orientation="vertical", cax=cax, ticks=a.ravel(), boundaries=np.arange(len(synset_names) + 1) - 0.5)
cax.set_yticklabels([s.capitalize() for s in synset_names], fontsize=6)
plt.savefig(os.path.join(savedir, "3dlt_umap_v24_colorbar.png"), bbox_inches='tight', dpi=300, transparent=True)

# Plot UMAP projections

In [ ]:
import umap

In [ ]:
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
umap_embeddings = umap_model.fit_transform(X_test)

In [ ]:
np.random.seed(42)
subset_indices = np.random.choice(len(umap_embeddings), size=10000, replace=False)

In [ ]:
# Plot UMAP embeddings colored by synset
plt.figure(figsize=(1.2, 1.5))
plt.scatter(umap_embeddings[subset_indices, 0], umap_embeddings[subset_indices, 1], 
            c=y_test_clf[subset_indices], cmap=cmap, s=1, alpha=0.7)
sns.despine(left=True, bottom=True)
plt.xticks([])
plt.yticks([])
plt.savefig(os.path.join(savedir, "3dlt_umap_v24_rplvic1.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
umap_embeddings_IL = umap_model.fit_transform(X_test_IL)

In [ ]:
subset_indices_IL = np.random.choice(len(umap_embeddings_IL), size=5000, replace=False)

In [ ]:
plt.figure(figsize=(1.2, 1.5))
plt.scatter(umap_embeddings_IL[subset_indices_IL, 0], umap_embeddings_IL[subset_indices_IL, 1], 
            c=y_test_clf_IL[subset_indices_IL], cmap=cmap, s=1, alpha=0.7)
sns.despine(left=True, bottom=True)
plt.xticks([])
plt.yticks([])
plt.savefig(os.path.join(savedir, "3dlt_umap_v24_IL1.png"), bbox_inches='tight', dpi=300, transparent=True)

# Plot linear projections onto coding direction

In [ ]:
def add_ax_dirs_and_scale(fig, is_3d=True, scale_annot="1.0", ax_annot=[r"$x$", r"$y$", r"$z$"]):

    fig.add_artist(plt.Line2D(
        [0.5, 0.9], [0.1, 0.1],
        transform=fig.transFigure,
        color="black",
        linewidth=2
    ))
    fig.text(
        0.7, -0.01,
        scale_annot,
        ha="center",
        va="bottom",
        fontsize=6,
        color="black"
    )
    if is_3d:
        ax_tri = fig.add_axes(
            [0.68, 0.8, 0.25, 0.25],
            projection="3d"
        )

        ax_tri.view_init(elev=15, azim=45)

        # Draw the three coordinate axes
        ax_tri.quiver(
            0, 0, 0,   1, 0, 0,
            color="black",
            arrow_length_ratio=0.2,
            linewidth=1
        )

        ax_tri.quiver(
            0, 0, 0,   0, 1, 0,
            color="black",
            arrow_length_ratio=0.2,
            linewidth=1
        )

        ax_tri.quiver(
            0, 0, 0,   0, 0, 1,
            color="black",
            arrow_length_ratio=0.2,
            linewidth=1
        )

        # Axis labels
        ax_tri.text(2, 0, 0, ax_annot[0], fontsize=6)
        ax_tri.text(0, 1.15, 0, ax_annot[1], fontsize=6)
        ax_tri.text(0, 0, 1.15, ax_annot[2], fontsize=6)

        # Make the inset clean
        ax_tri.set_xlim(0, 1.2)
        ax_tri.set_ylim(0, 1.2)
        ax_tri.set_zlim(0, 1.2)

        ax_tri.set_axis_off()
        return ax_tri
    else:
        ax_dir = fig.add_axes([0.70, 0.90, 0.22, 0.22])

        # X axis
        ax_dir.arrow(
            0, 0, 0.8, 0,
            head_width=0.12,
            head_length=0.12,
            fc="black",
            ec="black",
            linewidth=0.8,
            length_includes_head=True
        )

        # Y axis
        ax_dir.arrow(
            0, 0, 0, 0.8,
            head_width=0.12,
            head_length=0.12,
            fc="black",
            ec="black",
            linewidth=0.8,
            length_includes_head=True
        )

        # Labels
        ax_dir.text(1.1, 0, ax_annot[0], fontsize=7, ha="center", va="center")
        ax_dir.text(0, 1.1, ax_annot[1], fontsize=7, ha="center", va="center")

        # Clean inset
        ax_dir.set_xlim(-0.15, 1.1)
        ax_dir.set_ylim(-0.15, 1.1)
        ax_dir.set_aspect("equal")
        ax_dir.axis("off")

        return ax_dir

In [ ]:
def compute_projections(X, decoder):
    X = (X - X.mean(axis=0))
    w = decoder.coef_.T  # shape (latent_dim, output_dim)
    projections = X @ w   # shape (n_samples, output_dim)
    return projections

In [ ]:
decoders = np.load(os.path.join(repr_dir, "linear_decoders.npz"), allow_pickle=True)
decoders.keys()

In [ ]:
rpl_decoders = decoders["RPLVIC"].item()
rpl_decoders.keys()

In [ ]:
rplvic1_proj = compute_projections(X_test, rpl_decoders["ridge"][0])
rpl_vic1_rotz_proj = compute_projections(X_test, rpl_decoders["ridge_rot_z"][0])

In [ ]:
from repl.scripts.offline_eval_tlt12 import get_delta_features

## Plot colorbars

In [ ]:
def plot_proj_colorbar(x_label, vmin, vmax, is_circular=False, label_pos="bottom", save_path=None):
    a = np.array([[-1, 1]])
    plt.figure(figsize=(0.5, 0.1))
    cmap = cm.vikO if is_circular else cm.lipari
    img = plt.imshow(a, cmap=cmap)
    plt.gca().set_visible(False)
    cax = plt.axes([0.1, 0.2, 0.8, 0.6])
    plt.colorbar(orientation="horizontal", cax=cax)
    cax.xaxis.set_ticks_position('bottom')
    cax.xaxis.set_label_position(label_pos)
    cax.set_xlabel(x_label, fontsize=8)
    cax.set_xticks([-1, 1])
    cax.set_xticklabels([vmin, vmax], fontsize=6)
    cax.tick_params(length=0, pad=2)
    cax.xaxis.labelpad = 0
    if save_path is not None:
        plt.savefig(save_path, bbox_inches='tight', dpi=300, transparent=True)
    else:
        plt.show()

In [ ]:
plot_proj_colorbar(r'$r_1$', vmin="-1", vmax="1", is_circular=False,
                   label_pos="bottom", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_trans_x.pdf"))
plot_proj_colorbar(r'$r_2$', vmin="-1", vmax="1", is_circular=False,
                   label_pos="bottom", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_trans_y.pdf"))
plot_proj_colorbar(r'$r_3$', vmin="-1", vmax="1", is_circular=False,
                   label_pos="bottom", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_trans_z.pdf"))

In [ ]:
plot_proj_colorbar(r'$v_1$', vmin="-0.3", vmax="0.3", is_circular=False,
                   label_pos="bottom", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_delta_trans_x.pdf"))
plot_proj_colorbar(r'$v_2$', vmin="-0.3", vmax="0.3", is_circular=False,
                   label_pos="bottom", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_delta_trans_y.pdf"))
plot_proj_colorbar(r'$v_3$', vmin="-0.3", vmax="0.3", is_circular=False,
                   label_pos="bottom", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_delta_trans_z.pdf"))

In [ ]:
plot_proj_colorbar(r"$\theta^{\text{spot}}$", vmin="0", vmax="1", is_circular=False,
                   label_pos="bottom", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_spot_theta.pdf"))
plot_proj_colorbar(r"$\phi^{\text{spot}}$", vmin="-1", vmax="1", is_circular=True,
                   label_pos="bottom", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_spot_phi.pdf"))
plot_proj_colorbar("Spot hue", vmin="0", vmax="1", is_circular=True,
                   label_pos="bottom", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_spot_hue.pdf"))
plot_proj_colorbar("Floor hue", vmin="0", vmax="1", is_circular=True,
                   label_pos="bottom", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_floor_hue.pdf"))


In [ ]:
plot_proj_colorbar(r"$\theta_1$", vmin=r"$-\frac{\pi}{8}$", vmax=r"$\frac{\pi}{8}$",
                   is_circular=False, label_pos="top", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_rot_x.pdf"))
plot_proj_colorbar(r"$\theta_2$", vmin=r"$-\frac{\pi}{8}$", vmax=r"$\frac{\pi}{8}$",
                   is_circular=False, label_pos="top", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_rot_y.pdf"))
plot_proj_colorbar(r"$\theta_3$", vmin=r"$-\pi$", vmax=r"$\pi$",
                   is_circular=True, label_pos="top", save_path=os.path.join(savedir, "3dlt_proj_v24_colorbar_rot_z.pdf"))

## True projections

In [ ]:
np.random.seed(42)  # For reproducibility
sample_indices = np.random.choice(len(X_test), size=5000, replace=False)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr)
ax.scatter(deltas[sample_indices, 7], deltas[sample_indices, 8], deltas[sample_indices, 9], 
           c=deltas[sample_indices, 7], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.view_init(elev=15, azim=45)
ax.set_xlim(-0.2, 0.2)
ax.set_ylim(-0.2, 0.2)
ax.set_zlim(-0.2, 0.2)
add_ax_dirs_and_scale(fig, is_3d=True, scale_annot="0.3",
                      ax_annot=[r"$v_1$", r"$v_2$", r"$v_3$"])
fig.savefig(os.path.join(savedir, "3dlt_true_proj_delta_trans_x.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr)
ax.scatter(deltas[sample_indices, 7], deltas[sample_indices, 8], deltas[sample_indices, 9], 
           c=deltas[sample_indices, 8], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-0.2, 0.2)
ax.set_ylim(-0.2, 0.2)
ax.set_zlim(-0.2, 0.2)
add_ax_dirs_and_scale(fig, is_3d=True, scale_annot="0.3",
                      ax_annot=[r"$v_1$", r"$v_2$", r"$v_3$"])
fig.savefig(os.path.join(savedir, "3dlt_true_proj_delta_trans_y.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr)
ax.scatter(deltas[sample_indices, 7], deltas[sample_indices, 8], deltas[sample_indices, 9], 
           c=deltas[sample_indices, 9], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-0.2, 0.2)
ax.set_ylim(-0.2, 0.2)
ax.set_zlim(-0.2, 0.2)
add_ax_dirs_and_scale(fig, is_3d=True, scale_annot="0.3",
                      ax_annot=[r"$v_1$", r"$v_2$", r"$v_3$"])
fig.savefig(os.path.join(savedir, "3dlt_true_proj_delta_trans_z.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr)
ax.scatter(y_test_regr[sample_indices, 7], y_test_regr[sample_indices, 8], y_test_regr[sample_indices, 9], 
           c=y_test_regr[sample_indices, 7], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-1, 1)
add_ax_dirs_and_scale(fig, is_3d=True, scale_annot="1.0",
                      ax_annot=[r"$r_1$", r"$r_2$", r"$r_3$"])
fig.savefig(os.path.join(savedir, "3dlt_true_proj_trans_x.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr)
ax.scatter(y_test_regr[sample_indices, 7], y_test_regr[sample_indices, 8], y_test_regr[sample_indices, 9], 
           c=y_test_regr[sample_indices, 8], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-1, 1)
add_ax_dirs_and_scale(fig, is_3d=True, scale_annot="1.0",
                      ax_annot=[r"$r_1$", r"$r_2$", r"$r_3$"])
fig.savefig(os.path.join(savedir, "3dlt_true_proj_trans_y.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr)
ax.scatter(y_test_regr[sample_indices, 7], y_test_regr[sample_indices, 8], y_test_regr[sample_indices, 9], 
           c=y_test_regr[sample_indices, 9], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-1, 1)
add_ax_dirs_and_scale(fig, is_3d=True, scale_annot="1.0",
                      ax_annot=[r"$r_1$", r"$r_2$", r"$r_3$"])
fig.savefig(os.path.join(savedir, "3dlt_true_proj_trans_z.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.8, 0.8))
ax = fig.add_subplot(111)
ax.scatter(y_test_regr[sample_indices, 0], y_test_regr[sample_indices, 1], 
           c=y_test_regr[sample_indices, 0], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax_dir = add_ax_dirs_and_scale(fig, is_3d=False, scale_annot="1.0",
                      ax_annot=[r"$\hat\theta_1$", r"$\hat\theta_2$"])
ax_dir.set_position([-0.1, -0.1, 0.22, 0.22])
ax_dir.texts[0].set_position((1.2, 0.0))
ax_dir.texts[1].set_position((0.0, 1.2))
fig.savefig(os.path.join(savedir, "3dlt_true_proj_rot_x.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.8, 0.8))
ax = fig.add_subplot(111)
ax.scatter(y_test_regr[sample_indices, 0], y_test_regr[sample_indices, 1], 
           c=y_test_regr[sample_indices, 1], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax_dir = add_ax_dirs_and_scale(fig, is_3d=False, scale_annot="1.0",
                      ax_annot=[r"$\hat\theta_1$", r"$\hat\theta_2$"])
ax_dir.set_position([-0.1, -0.1, 0.22, 0.22])
ax_dir.texts[0].set_position((1.2, 0.0))
ax_dir.texts[1].set_position((0.0, 1.2))
fig.savefig(os.path.join(savedir, "3dlt_true_proj_rot_y.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.8, 0.8))
ax = fig.add_subplot(111)
ax.scatter(np.cos(y_test_regr[sample_indices, 2]*np.pi), np.sin(y_test_regr[sample_indices, 2]*np.pi), 
           c=y_test_regr[sample_indices, 2], cmap=cm.vikO, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax_dir = add_ax_dirs_and_scale(fig, is_3d=False, scale_annot="1.0",
                      ax_annot=[r"$\cos(\theta_3)$", r"$\sin(\theta_3)$"])
# move ax_dir annotations to the right of the axes
cos_text = ax_dir.texts[0]
sin_text = ax_dir.texts[1]
sin_text.set_rotation(90)
sin_text.set_position((0, 2.1))
ax_dir.set_position([-0.1, -0.1, 0.22, 0.22])
cos_text.set_position((2.1, 0))
fig.savefig(os.path.join(savedir, "3dlt_true_proj_rot_z.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(y_test_regr[sample_indices, 4], 
           np.cos(y_test_regr[sample_indices, 5]*2*np.pi), 
           np.sin(y_test_regr[sample_indices, 5]*2*np.pi), 
           c=y_test_regr[sample_indices, 4], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_ylim([-1, 1])
ax.set_xlim([-1, 1])
ax.set_zlim([-1, 1])
tri_ax = add_ax_dirs_and_scale(fig, is_3d=True, scale_annot="0.5",
                      ax_annot=[r"$\theta^{\text{spot}}$", r"$\cos(\phi^{\text{spot}})$", r"$\sin(\phi^{\text{spot}})$"])
# move tri_ax slightly to the left
tri_ax.set_position([0.4, 0.8, 0.25, 0.25])
theta_text = tri_ax.texts[0]
theta_text.set_position((3, 0, 0))
fig.savefig(os.path.join(savedir, "3dlt_true_proj_spot_theta.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(y_test_regr[sample_indices, 4], 
           np.cos(y_test_regr[sample_indices, 5]*2*np.pi), 
           np.sin(y_test_regr[sample_indices, 5]*2*np.pi), 
           c=y_test_regr[sample_indices, 5], cmap=cm.vikO, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_ylim([-1, 1])
ax.set_xlim([-1, 1])
ax.set_zlim([-1, 1])
tri_ax = add_ax_dirs_and_scale(fig, is_3d=True, scale_annot="0.5",
                      ax_annot=[r"$\theta^{\text{spot}}$", r"$\cos(\phi^{\text{spot}})$", r"$\sin(\phi^{\text{spot}})$"])
# move tri_ax slightly to the left
tri_ax.set_position([0.4, 0.8, 0.25, 0.25])
theta_text = tri_ax.texts[0]
theta_text.set_position((3, 0, 0))
fig.savefig(os.path.join(savedir, "3dlt_true_proj_spot_phi.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.85, 0.85), constrained_layout=True)
ax = fig.add_subplot(111)
ax.scatter(np.cos(y_test_regr[sample_indices, 3]*2*np.pi), 
           np.sin(y_test_regr[sample_indices, 3]*2*np.pi), 
           c=y_test_regr[sample_indices, 3], cmap=cm.vikO, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_ylim([-1.5, 1.5])
ax.set_xlim([-1.5, 1.5])
ax_dir = add_ax_dirs_and_scale(fig, is_3d=False, scale_annot="1.5",
                      ax_annot=[r"$\cos(\text{hue})$", r"$\sin(\text{hue})$"])
ax_dir.set_position([0.5, 0.9, 0.25, 0.25])
cos_text = ax_dir.texts[0]
cos_text.set_position((2., 0))
fig.savefig(os.path.join(savedir, "3dlt_true_proj_floor_hue.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.85, 0.85), constrained_layout=True)
ax = fig.add_subplot(111)
ax.scatter(np.cos(y_test_regr[sample_indices, 6]*2*np.pi), 
           np.sin(y_test_regr[sample_indices, 6]*2*np.pi), 
           c=y_test_regr[sample_indices, 6], cmap=cm.vikO, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_ylim([-1.5, 1.5])
ax.set_xlim([-1.5, 1.5])
ax_dir = add_ax_dirs_and_scale(fig, is_3d=False, scale_annot="1.5",
                      ax_annot=[r"$\cos(\text{hue})$", r"$\sin(\text{hue})$"])
ax_dir.set_position([0.5, 0.9, 0.25, 0.25])
cos_text = ax_dir.texts[0]
cos_text.set_position((2., 0))
fig.savefig(os.path.join(savedir, "3dlt_true_proj_spot_hue.png"), bbox_inches='tight', dpi=300, transparent=True)

## RPLVIC projections

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr)
ax.scatter(rplvic1_proj[sample_indices, 15], rplvic1_proj[sample_indices, 16], rplvic1_proj[sample_indices, 17], 
           c=deltas[sample_indices, 7], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.view_init(elev=15, azim=45)
ax.set_xlim(-0.2, 0.2)
ax.set_ylim(-0.2, 0.2)
ax.set_zlim(-0.2, 0.2)

fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_delta_trans_x.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr)
ax.scatter(rplvic1_proj[sample_indices, 15], rplvic1_proj[sample_indices, 16], rplvic1_proj[sample_indices, 17], 
           c=deltas[sample_indices, 8], cmap=cm.lipari, s=1, alpha=0.7)
ax.view_init(elev=15, azim=45)
ax.set_axis_off()
ax.set_ylim([-0.2, 0.2])
ax.set_xlim([-0.2, 0.2])
ax.set_zlim([-0.2, 0.2])
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_delta_trans_y.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr)
ax.scatter(rplvic1_proj[sample_indices, 15], rplvic1_proj[sample_indices, 16], rplvic1_proj[sample_indices, 17], 
           c=deltas[sample_indices, 9], cmap=cm.lipari, s=1, alpha=0.7)
ax.view_init(elev=15, azim=45)
ax.set_axis_off()
ax.set_ylim([-0.2, 0.2])
ax.set_xlim([-0.2, 0.2])
ax.set_zlim([-0.2, 0.2])
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_delta_trans_z.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(rplvic1_proj[sample_indices, 3], rplvic1_proj[sample_indices, 4], rplvic1_proj[sample_indices, 5], 
           c=y_test_regr[sample_indices, 7], cmap=cm.lipari, s=1, alpha=0.7)
ax.view_init(elev=15, azim=45)
ax.set_axis_off()
ax.set_ylim([-1, 1])
ax.set_xlim([-1, 1])
ax.set_zlim([-1, 1])
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_trans_x.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(rplvic1_proj[sample_indices, 3], rplvic1_proj[sample_indices, 4], rplvic1_proj[sample_indices, 5], 
           c=y_test_regr[sample_indices, 8], cmap=cm.lipari, s=1, alpha=0.7)
ax.view_init(elev=15, azim=45)
ax.set_axis_off()
ax.set_ylim([-1, 1])
ax.set_xlim([-1, 1])
ax.set_zlim([-1, 1])
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_trans_y.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(rplvic1_proj[sample_indices, 3], rplvic1_proj[sample_indices, 4], rplvic1_proj[sample_indices, 5], 
           c=y_test_regr[sample_indices, 9], cmap=cm.lipari, s=1, alpha=0.7)
ax.view_init(elev=15, azim=45)
ax.set_axis_off()
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-1, 1)
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_trans_z.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.8, 0.8))
ax = fig.add_subplot(111)
ax.scatter(rplvic1_proj[sample_indices, 0], rplvic1_proj[sample_indices, 1], 
           c=y_test_regr[sample_indices, 0], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_axis_off()

fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_rot_x.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.8, 0.8))
ax = fig.add_subplot(111)
ax.scatter(rplvic1_proj[sample_indices, 0], rplvic1_proj[sample_indices, 1], 
           c=y_test_regr[sample_indices, 1], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_rot_y.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.8, 0.8))
ax = fig.add_subplot(111)
ax.scatter(rpl_vic1_rotz_proj[sample_indices, -1], rpl_vic1_rotz_proj[sample_indices, -2], 
           c=y_test_regr[sample_indices, 2], cmap=cm.vikO, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_rot_z.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(rplvic1_proj[sample_indices, 2], rplvic1_proj[sample_indices, 10], rplvic1_proj[sample_indices, 7], 
           c=y_test_regr[sample_indices, 4], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.view_init(elev=15, azim=45)
ax.set_ylim([-1, 1])
ax.set_xlim([-1, 1])
ax.set_zlim([-1, 1])
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_spot_theta.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(rplvic1_proj[sample_indices, 2], rplvic1_proj[sample_indices, 10], rplvic1_proj[sample_indices, 7], 
           c=y_test_regr[sample_indices, 5], cmap=cm.vikO, s=1, alpha=0.7)
ax.set_axis_off()
ax.view_init(elev=15, azim=45)
ax.set_ylim([-1, 1])
ax.set_xlim([-1, 1])
ax.set_zlim([-1, 1])
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_spot_phi.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.85, 0.85), constrained_layout=True)
ax = fig.add_subplot(111)
ax.scatter(rplvic1_proj[sample_indices, 6], rplvic1_proj[sample_indices, 9], 
           c=y_test_regr[sample_indices, 3], cmap=cm.vikO, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_ylim([-1.5, 1.5])
ax.set_xlim([-1.5, 1.5])
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_floor_hue.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.85, 0.85), constrained_layout=True)
ax = fig.add_subplot(111)
ax.scatter(rplvic1_proj[sample_indices, 8], rplvic1_proj[sample_indices, 11], c=y_test_regr[sample_indices, 6], cmap=cm.vikO, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_ylim([-1.5, 1.5])
ax.set_xlim([-1.5, 1.5])
fig.savefig(os.path.join(savedir, "3dlt_rplvic1_proj_spot_hue.png"), bbox_inches='tight', dpi=300, transparent=True)

## IL projections

In [ ]:
il_decoders = decoders["IL"].item()

In [ ]:
il1_proj = compute_projections(X_test_IL, il_decoders["ridge"][0])
il1_rotz_proj = compute_projections(X_test_IL, il_decoders["ridge_rot_z"][0])

In [ ]:
np.random.seed(42)
sample_indices = np.random.choice(len(X_test_IL), size=5000, replace=False)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr_IL)
ax.scatter(il1_proj[sample_indices, 15], il1_proj[sample_indices, 16], il1_proj[sample_indices, 17], 
           c=deltas[sample_indices, 7], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.view_init(elev=15, azim=45)
ax.set_ylim(-0.2, 0.2)
ax.set_xlim(-0.2, 0.2)
ax.set_zlim(-0.2, 0.2)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_delta_trans_x.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr_IL)
ax.scatter(il1_proj[sample_indices, 15], il1_proj[sample_indices, 16], il1_proj[sample_indices, 17], 
           c=deltas[sample_indices, 8], cmap=cm.lipari, s=1, alpha=0.7)
ax.view_init(elev=15, azim=45)
ax.set_axis_off()
ax.set_ylim(-0.2, 0.2)
ax.set_xlim(-0.2, 0.2)
ax.set_zlim(-0.2, 0.2)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_delta_trans_y.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
deltas = get_delta_features(y_test_regr_IL)
ax.scatter(il1_proj[sample_indices, 15], il1_proj[sample_indices, 16], il1_proj[sample_indices, 17], 
           c=deltas[sample_indices, 9], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.view_init(elev=15, azim=45)
ax.set_ylim(-0.2, 0.2)
ax.set_xlim(-0.2, 0.2)
ax.set_zlim(-0.2, 0.2)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_delta_trans_z.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(il1_proj[sample_indices, 3], il1_proj[sample_indices, 4], il1_proj[sample_indices, 5], 
           c=y_test_regr_IL[sample_indices, 7], cmap=cm.lipari, s=1, alpha=0.7)
ax.view_init(elev=15, azim=45)
ax.set_axis_off()
ax.set_ylim(-1, 1)
ax.set_xlim(-1, 1)
ax.set_zlim(-1, 1)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_trans_x.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(il1_proj[sample_indices, 3], il1_proj[sample_indices, 4], il1_proj[sample_indices, 5], 
           c=y_test_regr_IL[sample_indices, 8], cmap=cm.lipari, s=1, alpha=0.7)
ax.view_init(elev=15, azim=45)
ax.set_axis_off()
ax.set_ylim(-1, 1)
ax.set_xlim(-1, 1)
ax.set_zlim(-1, 1)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_trans_y.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(il1_proj[sample_indices, 3], il1_proj[sample_indices, 4], il1_proj[sample_indices, 5], 
           c=y_test_regr_IL[sample_indices, 9], cmap=cm.lipari, s=1, alpha=0.7)
ax.view_init(elev=15, azim=45)
ax.set_axis_off()
ax.set_ylim(-1, 1)
ax.set_xlim(-1, 1)
ax.set_zlim(-1, 1)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_trans_z.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.8, 0.8))
ax = fig.add_subplot(111)
ax.scatter(il1_proj[sample_indices, 0], il1_proj[sample_indices, 1], 
           c=y_test_regr_IL[sample_indices, 0], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_rot_x.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.8, 0.8))
ax = fig.add_subplot(111)
ax.scatter(il1_proj[sample_indices, 0], il1_proj[sample_indices, 1], 
           c=y_test_regr_IL[sample_indices, 1], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_rot_y.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.8, 0.8))
ax = fig.add_subplot(111)
ax.scatter(il1_rotz_proj[sample_indices, -1], il1_rotz_proj[sample_indices, -2], 
           c=y_test_regr_IL[sample_indices, 2], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_ylim([-1.1, 1.1])
ax.set_xlim([-1.1, 1.1])
# sns.despine()
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_rot_z.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(il1_proj[sample_indices, 2], il1_proj[sample_indices, 7], il1_proj[sample_indices, 10], 
           c=y_test_regr_IL[sample_indices, 4], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.view_init(elev=15, azim=45)
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-1, 1)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_spot_theta.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(1, 1), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(il1_proj[sample_indices, 2], il1_proj[sample_indices, 7], il1_proj[sample_indices, 10], 
           c=y_test_regr_IL[sample_indices, 5], cmap=cm.lipari, s=1, alpha=0.7)
ax.set_axis_off()
ax.view_init(elev=15, azim=45)
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-1, 1)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_spot_phi.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.85, 0.85), constrained_layout=True)
ax = fig.add_subplot(111)
ax.scatter(il1_proj[sample_indices, 6], il1_proj[sample_indices, 9], 
           c=y_test_regr_IL[sample_indices, 3], cmap=cm.vikO, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_floor_hue.png"), bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
fig = plt.figure(figsize=(0.85, 0.85), constrained_layout=True)
ax = fig.add_subplot(111)
ax.scatter(il1_proj[sample_indices, 8], il1_proj[sample_indices, 11], 
           c=y_test_regr_IL[sample_indices, 6], cmap=cm.vikO, s=1, alpha=0.7)
ax.set_axis_off()
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
fig.savefig(os.path.join(savedir, "3dlt_il1_proj_spot_hue.png"), bbox_inches='tight', dpi=300, transparent=True)